# 第 10 讲练习：图、DFS 与 BFS

每道题先回答四个问题：节点是什么？邻居怎样产生？何时标记 visited？什么时候返回？

In [1]:
from collections import deque

## 1. 概念题

1. 邻接表、邻接矩阵和边列表分别怎样表示图？稀疏图通常更适合哪一种？
2. 无向边 `(a, b)` 转成邻接表时为什么要加入两次？
3. DFS 和 BFS 都能解决哪些问题？为什么无权图最短边数通常使用 BFS？
4. 为什么 DFS 通常进入节点后立即标记，BFS 通常在入队时标记？
5. 图不连通时，为什么只从节点 `0` 搜索不够？
6. 图搜索明明有嵌套循环，为什么复杂度通常是 `O(V + E)`？
7. 无向图和有向图检测环时，visited 的含义有什么不同？

在这里作答。

答：1. 邻接表写出每个节点的邻居有哪些。邻接矩阵使用一个n*n的矩阵中的内容来表示两个节点是否有连接。边列表列出图中的所有边。稀疏图更适合邻接表。   
2.因为a的邻居有b，b的邻居有a   
3.DFS解决探索可能的路径、检测环。BFS按层遍历，可以寻找最短路径、层序遍历。如果使用DFS就得探索完全图所有的路径，复杂度高   
4.DFS立即标记是为了避免陷入循环。BFS入队标记是为了避免用一个节点多次入队   
5.不联通那么就找不到其他的点   
6.外层循环：每个点最多被处理一次，执行次数是V。内部循环：执行总次数是2E。最后的复杂度是相加不是相乘   
7.无向图标记这个点是否到达过。有向图需要用三色标记。   

## 2. 建图与 DFS 可达集合

实现两个函数：

- `build_undirected_graph(n, edges)`：把 `0` 到 `n-1` 的无向边列表转换为邻接表；
- `reachable_nodes(graph, start)`：使用 DFS 返回从 `start` 可以到达的节点集合。

邻接表中必须保留没有边的节点。

In [7]:
def build_undirected_graph(n, edges): #边列表 转 邻接表
    graph = [[] for _ in range(n)]
    for x,y in edges:
        graph[x].append(y)
        graph[y].append(x)
    return graph

def reachable_nodes(graph, start):
    result = set()
    def dfs(node):
        result.add(node)
        for x in graph[node]:
            if x not in result:
                dfs(x)

    dfs(start)
    return result


In [4]:
graph

[[1, 2], [0, 3], [0], [1], [5], [4]]

In [8]:
graph = build_undirected_graph(6, [(0,1), (0,2), (1,3), (4,5)])
assert len(graph) == 6
assert set(graph[0]) == {1,2}
assert set(graph[1]) == {0,3}
assert reachable_nodes(graph, 0) == {0,1,2,3}
assert reachable_nodes(graph, 4) == {4,5}
assert reachable_nodes(build_undirected_graph(1, []), 0) == {0}

## 3. 连通分量数量

给定节点数 `n` 和无向边列表，返回连通分量数量。你可以复用建图函数。

思考：什么时候给答案加一？为什么一次 DFS 内部不能重复加一？

In [11]:
def count_components(n, edges):
    if edges == []:
        return n
    graph = build_undirected_graph(n,edges) #转成邻接图


    def dfs(node):
        visit.add(node)
        for i in graph[node]:
            if i not in visit:
                dfs(i)


    cnt = 0
    visit = set()

    for x in range(n):
        if x not in visit:
            cnt += 1
            dfs(x)
    return cnt

In [12]:
assert count_components(0, []) == 0
assert count_components(1, []) == 1
assert count_components(5, [(0,1), (1,2), (3,4)]) == 2
assert count_components(5, []) == 5
assert count_components(4, [(0,1), (1,2), (2,3), (3,0)]) == 1

## 4. BFS 最短路径长度

给定无权图的邻接表，返回 `start` 到 `target` 的最少边数；不可达返回 `-1`。

要求在节点入队时记录距离并加入 visited，而不是出队时才标记。

In [15]:
def shortest_path_length(graph, start, target):
    que = deque()
    visit = {start:0}

    que.append(start)

    while que:
        cur = que.popleft()
        for x in graph[cur]:
            if x not in visit:
                que.append(x)
                visit[x] = visit[cur]+1
            if x == target:
                return visit[x]
    
    return -1

In [16]:
graph = {
    'A':['B','C'],
    'B':['A','D'],
    'C':['A','D','E'],
    'D':['B','C','F'],
    'E':['C'],
    'F':['D'],
    'G':[]
}
assert shortest_path_length(graph, 'A', 'A') == 0
assert shortest_path_length(graph, 'A', 'F') == 3
assert shortest_path_length(graph, 'B', 'E') == 3
assert shortest_path_length(graph, 'A', 'G') == -1

## 5. 岛屿数量

`'1'` 表示陆地，`'0'` 表示水；只有上下左右相邻的陆地连通。返回岛屿数量。

本题不允许修改输入网格，请使用 `visited`。先检查边界，再读取相邻格子。

In [26]:
def num_islands(grid):
    if grid == []:
        return 0
    row = len(grid)
    col = len(grid[0])
    ans = 0
    visit = set()

    dir = [(0,1),(0,-1),(1,0),(-1,0)]

    def dfs(x,y):
        visit.add((x,y))
        for dr,dc in dir:
            cr = x+dr
            cc = y+dc
            if 0 <= cr < row and 0 <=cc <col:
                if (cr,cc) not in visit and grid[cr][cc] == '1':
                    dfs(cr,cc)

    for r in range(row):
        for c in range(col):
            if grid[r][c] == '1' and (r,c) not in visit:
                ans += 1
                dfs(r,c)
    
    return ans

In [27]:
grid1 = [
    ['1','1','0','0'],
    ['1','0','0','1'],
    ['0','0','1','1'],
]
grid1_copy = [row.copy() for row in grid1]
assert num_islands(grid1) == 2
assert grid1 == grid1_copy  # 不修改输入
assert num_islands([]) == 0
assert num_islands([['0','0'], ['0','0']]) == 0
assert num_islands([['1','0'], ['0','1']]) == 2

## 6. 无向图检测环

判断无向图中是否存在环。图可能不连通。

递归 DFS 除了当前节点，还要接收 `parent`。遇到已访问邻居时，为什么只有它不是父节点才说明有环？

In [ ]:
def has_cycle_undirected(graph):
    visit = set()

    def dfs(node,parent = None):
        # 遍历node的所有邻居，查看是否有拜访过的非父节点
        # 如果拜访过，说明有环，返回True
        # 如果没问题，返回False
        # 注释很重要，要明白返回值的意义,True表示的到底是有环还是没有环
        visit.add(node)
        for x in graph[node]:
            if x not in visit: #没有拜访过，加入栈
                if dfs(x,node):
                    return True #过程中发现环，退出
        
            
            elif x == parent: #刚刚经过的节点，跳过
                continue
            
            else:
                return True #拜访过的节点不是父节点，说明有环
        
        return False #没有环
    
    for x in graph:
        if x not in visit:
            if dfs(x):
                return True
    
    return False

In [31]:
assert has_cycle_undirected({0:[1], 1:[0,2], 2:[1]}) is False
assert has_cycle_undirected({0:[1,2], 1:[0,2], 2:[0,1]}) is True
assert has_cycle_undirected({0:[], 1:[]}) is False
assert has_cycle_undirected({0:[1], 1:[0], 2:[3,4], 3:[2,4], 4:[2,3]}) is True

## 7. 课程表：拓扑排序

`prerequisites[i] = [course, prerequisite]` 表示先学 `prerequisite` 才能学 `course`。判断能否完成所有课程。

使用 Kahn 算法：建立 `prerequisite -> course` 的边，维护入度，并从所有入度为 0 的课程开始 BFS。

步骤1： prerequisites本质是边图，要建立起邻接图graph
步骤2：统计所有入度
步骤3：入度=0的节点入队，开始BFS

关键是找准，什么时候入队，degree怎么变化

In [34]:
from collections import defaultdict

def can_finish(num_courses, prerequisites):

    graph = [[] for _ in range(num_courses)]
    for x in prerequisites:
        graph[x[1]].append(x[0])

    degree = [0]*num_courses
    for x in graph: #x是一个list
        for y in x:
            degree[y] += 1

    que = deque()
    visit = set()



    for x in range(num_courses): #degree是一个用list表示的字典
        if degree[x] == 0:
            que.append(x)
            visit.add(x)

    
    while que:
        cur = que.popleft()
        for x in graph[cur]:
            if x not in visit:
                degree[x] -= 1
                if degree[x] == 0:
                    que.append(x)
                    visit.add(x)
    
    return len(visit) == num_courses

kahn算法不需要visit，因为只有degree = 0时才会入队

In [35]:
assert can_finish(2, [[1,0]]) is True
assert can_finish(2, [[1,0], [0,1]]) is False
assert can_finish(4, [[1,0], [2,0], [3,1], [3,2]]) is True
assert can_finish(3, []) is True
assert can_finish(3, [[0,1], [1,2], [2,1]]) is False

## 8. 挑战：克隆图

深拷贝一个可能含环的无向连通图。每个原节点必须只对应一个新节点，副本中的连接关系与原图一致。

使用字典保存 `原节点对象 -> 新节点对象`。为什么创建副本后必须立即写入字典，再递归邻居？

In [ ]:
class Node:
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = [] if neighbors is None else neighbors

def clone_graph(node):
    pass

In [ ]:
def graph_snapshot(start):
    if start is None:
        return {}
    result = {}
    queue = deque([start])
    visited = {start}
    while queue:
        node = queue.popleft()
        result[node.val] = sorted(neighbor.val for neighbor in node.neighbors)
        for neighbor in node.neighbors:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return result

a, b, c = Node(1), Node(2), Node(3)
a.neighbors = [b, c]
b.neighbors = [a, c]
c.neighbors = [a, b]
copy_a = clone_graph(a)
assert graph_snapshot(copy_a) == graph_snapshot(a)
assert copy_a is not a
assert all(new is not old for new in copy_a.neighbors for old in [a,b,c])
assert clone_graph(None) is None

# 累计复习

## 9. 堆复习：最小的 `k` 个数

返回数组中最小的 `k` 个数，顺序不限；`k == 0` 返回空列表。不能完整排序。

提示：要保留最小的 `k` 个，堆满时最应该淘汰谁？Python 中怎样模拟这个堆？

In [ ]:
import heapq

def k_smallest(nums, k):
    pass

In [ ]:
assert k_smallest([3,2,1], 0) == []
assert sorted(k_smallest([3,2,1], 2)) == [1,2]
assert sorted(k_smallest([7,10,4,3,20,15], 3)) == [3,4,7]
assert sorted(k_smallest([-1,-5,2,0], 4)) == [-5,-1,0,2]

## 10. 二叉树复习：判断对称

判断二叉树是否关于中心轴对称。辅助函数比较的是哪两个方向？

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def is_symmetric(root):
    pass

In [ ]:
assert is_symmetric(None) is True
assert is_symmetric(TreeNode(1)) is True
symmetric = TreeNode(1, TreeNode(2, TreeNode(3), TreeNode(4)), TreeNode(2, TreeNode(4), TreeNode(3)))
assert is_symmetric(symmetric) is True
asymmetric = TreeNode(1, TreeNode(2, None, TreeNode(3)), TreeNode(2, None, TreeNode(3)))
assert is_symmetric(asymmetric) is False

## 11. 总结

1. 每道图题中的节点和边分别表示什么？
2. 哪些题需要外层扫描所有节点或格子？
3. DFS 与 BFS 中 visited 分别在什么时候设置？
4. 无向环检测为什么需要 parent？
5. 拓扑排序的队列中始终保存什么？
6. 克隆图的字典为什么同时承担 visited 和映射两种职责？
7. 两道累计复习题能否不看旧讲义完成？